# 02 · Data preparation

Supports **Section 6.4** of the thesis. The notebook builds the inputs of the forecasting models from the raw
M5 files and checks them against the raw data. The construction itself lives in `src/features.py`, so that
notebooks 04 and 05, and a run on a GPU machine, build exactly the same design matrices.

**Design**
- **One model for the 7-day horizon.** A training example is a triple (series, forecast origin t, horizon h):
  the inputs are what is known on day t plus the horizon h = 1..7, the target is the sales on day t + h.
  The model forecasts every horizon directly, so no forecast is fed back into it.
- **Two stages.** Tuning uses targets up to 24.04.2016. and forecasts the validation window 25.04.–22.05.2016.
  The final models are refitted on targets up to 22.05.2016. and forecast the test window 23.05.–19.06.2016.
  The test sales are not loaded here.
- **Rolling evaluation.** Each 28-day window is forecast 7 days at a time from 4 weekly origins (validation:
  24.04.2016., 01.05.2016., 08.05.2016., 15.05.2016.). The model is not refitted between the origins, but its inputs use
  the sales observed up to each origin.
- **Training origins** every `stride` days back from the stage's forecast origin, from each series' first
  sale. With stride 7 all origins fall on a Sunday, the weekday of both forecast origins, so each horizon
  has the same target weekday in training and in the forecast, and every day is the target of one origin.
- **Missing, not zero.** Days before a series' first sale and Christmas Day (stores closed) are missing in
  the sales history, and rows whose target day is Christmas are dropped. XGBoost handles missing inputs.

**Input groups** (the feature sets of the design are L, L+E, L+E+S, L+E+S+ID and L+E+S+ID+H; the horizon h is always an input):

| Group | Inputs |
|---|---|
| L | sales on days t, t − 1, …, t − 55 |
| E | known in advance for the target day: weekday, event, days to and since the nearest event, SNAP, relative price, discount |
| S | from the sales history up to t: rolling means (7, 28, 56, 182 days), zero share (28, 182 days), mean non-zero size (56 days), days since the last sale, age of the series |
| ID | item, department, category, store, state; kept apart from S, so that series identity and sales-derived inputs can be evaluated separately |
| H | sales of higher levels of the hierarchy up to t: mean daily sales over the last 7 and 28 days and their ratio, for the item in all stores, the department in the store and the whole store |

In [1]:
import copy
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import features as F
import m5_data as m5

pd.set_option("display.float_format", "{:,.3f}".format)
RESULTS = ROOT / "results" / "data_prep"
RESULTS.mkdir(parents=True, exist_ok=True)
PANEL_DIR = m5.PROCESSED_DIR / "panel"

## 1. Panel

All inputs per series and day, built once from the raw files and saved to `data/processed/panel/`
(not tracked by git).

In [2]:
panel = F.build_panel()
panel.save(PANEL_DIR)

y = panel.y[:, 1:]
pre_intro = np.arange(1, m5.VALID_END + 1)[None, :] < panel.first_sale[:, None]
assert panel.y.shape[1] - 1 == m5.VALID_END, "the test window must stay sealed"
print(f"series        : {len(panel.ids):,}")
print(f"sales history: {m5.window_name(1, m5.VALID_END)}; test window {m5.window_name(m5.VALID_END + 1, m5.TEST_END)} not loaded")
print(f"missing cells : {np.isnan(y).mean():.1%} (before the first sale {pre_intro.mean():.1%}, "
      f"Christmas {(np.isnan(y) & ~pre_intro).mean():.2%})")
print(f"price weeks   : {panel.price['rel_price'].shape[1]}")
print(f"saved to      : {PANEL_DIR.relative_to(ROOT)}")

series        : 30,490
sales history: 29.01.2011.–22.05.2016.; test window 23.05.–19.06.2016. not loaded


missing cells : 21.1% (before the first sale 20.9%, Christmas 0.21%)
price weeks   : 282
saved to      : data\processed\panel


## 2. Inputs

89 inputs: the horizon h and 88 inputs in five groups. The 56 lags are listed in `results/data_prep/features.csv`; the table shows the others.

In [3]:
listed = ~np.isnan(panel.price["rel_price"])
after_first = np.arange(listed.shape[1])[None, :] >= np.argmax(listed, axis=1)[:, None]
print(f"item-store weeks after the first price week without a price: {(~listed & after_first).sum():,} "
      f"of {after_first.sum():,}")

item-store weeks after the first price week without a price: 0 of 6,841,121


In [4]:
features = F.feature_table()
features.to_csv(RESULTS / "features.csv", index=False)
print(features.groupby("group", sort=False).size().rename("inputs").to_string())
features[features.group != "L"].set_index("feature")

group
h      1
L     56
E      9
S      9
ID     5
H      9


,group,description
feature,,
h,h,horizon: days from the origin t to the target ...
wday,E,"weekday of the target day (M5 coding, 1 = Satu..."
event,E,"event on the target day (30 named events, cate..."
event_type,E,"event type on the target day: Cultural, Nation..."
days_to_event,E,days from the target day to the next event (0 ...
days_since_event,E,days from the previous event to the target day...
snap,E,SNAP day in the store's state on the target da...
rel_price,E,price in the target week / the series' median ...
price_change,E,price in the target week / price in the previo...


## 3. Number of training examples

Training rows per stage and origin stride: with all 7 horizons of every origin, and with 2 horizons drawn at
random per origin, which is what the global models use to fit in the memory of the GPU. Only pairs with
t + h ≤ T are counted. Memory is that of 80 float32 inputs (the input set L+E+S+ID; group H adds 9 more).

In [5]:
rows = []
for stage, T in F.STAGES.items():
    for stride in (1, 7):
        series, t = F.training_origins(panel, stage, stride)
        for per_origin in (None, 2):
            r, h = F.horizon_rows(t.size, per_origin)
            n = int((t[r].astype(int) + h <= T).sum())
            rows.append({"stage": stage, "origin": m5.date_of(T), "stride": stride,
                         "horizons per origin": per_origin or F.HORIZON, "rows": n, "memory_GB": n * 80 * 4 / 1e9})
design_size = pd.DataFrame(rows)
design_size.to_csv(RESULTS / "design_size.csv", index=False)
design_size.style.format({"rows": "{:,}", "memory_GB": "{:.1f}"}).hide(axis="index")

stage,origin,stride,horizons per origin,rows,memory_GB
valid,24.04.2016.,1,7,"320,743,780",102.6
valid,24.04.2016.,1,2,"91,641,312",29.3
valid,24.04.2016.,7,7,"45,839,759",14.7
valid,24.04.2016.,7,2,"13,097,074",4.2
test,22.05.2016.,1,7,"326,719,820",104.6
test,22.05.2016.,1,2,"93,348,759",29.9
test,22.05.2016.,7,7,"46,693,479",14.9
test,22.05.2016.,7,2,"13,340,994",4.3


## 4. Design matrix

Validation stage with weekly origins and one random horizon per origin, which keeps the matrix small enough
for this machine and still covers all horizons.

In [6]:
series, t = F.training_origins(panel, "valid", stride=7)
table = F.origin_table(panel, series, t)
rows, h = F.horizon_rows(len(table), per_origin=1)
X, y, keep = F.design(panel, table, h, stage="valid", rows=rows)
rows, h = rows[keep], h[keep]
print(f"origins: {len(table):,}")
print(f"matrix : {X.shape[0]:,} rows x {X.shape[1]} inputs ({X.memory_usage(deep=True).sum() / 1e9:.1f} GB), "
      f"zero targets {np.mean(y == 0):.1%}")
print("rows by horizon:", pd.Series(h).value_counts().sort_index().to_dict())

origins: 6,548,537
matrix : 6,530,772 rows x 80 inputs (2.0 GB), zero targets 59.5%
rows by horizon: {1: 933665, 2: 932270, 3: 932338, 4: 930091, 5: 929923, 6: 936925, 7: 935560}


In [7]:
num = [c for c in F.E_COLS + F.S_COLS if c not in ("event", "event_type")]
summary = pd.DataFrame(
    {
        "group": [F.FEATURE_INFO[c][0] for c in num],
        "missing": X[num].isna().mean(),
        "mean": X[num].mean(),
        "median": X[num].median(),
        "max": X[num].max(),
    }
)
lag_missing = X[F.LAG_COLS].isna().mean()
print(f"missing lags: lag_1 {lag_missing['lag_1']:.2%}, lag_56 {lag_missing['lag_56']:.2%}")
summary.to_csv(RESULTS / "input_summary.csv")
summary

missing lags: lag_1 0.25%, lag_56 3.68%


,group,missing,mean,median,max
wday,E,0.000,3.995,4.000,7.000
days_to_event,E,0.000,9.517,7.000,50.000
days_since_event,E,0.001,9.460,7.000,50.000
snap,E,0.000,0.330,0.000,1.000
rel_price,E,0.000,0.999,1.000,8.613
price_change,E,0.002,0.001,0.000,897.000
discount,E,0.000,0.011,0.000,0.999
mean_7,S,0.000,1.437,0.500,568.000
mean_28,S,0.000,1.440,0.536,418.179
mean_56,S,0.000,1.443,0.554,398.000


## 5. Checks

**Against the raw files.** 2,000 random rows of the matrix are recomputed directly from the raw CSV
files, without `src/features.py`: lags, target, rolling inputs, event, SNAP and relative price.

In [8]:
sales = m5.load_sales(last_day=m5.VALID_END)
raw = sales[m5.day_cols(1, m5.VALID_END)].to_numpy(float)
calendar = m5.load_calendar().set_index("d_num")
christmas = set(calendar.index[(calendar.date.dt.month == 12) & (calendar.date.dt.day == 25)])
prices = m5.load_prices().set_index(["store_id", "item_id", "wm_yr_wk"])["sell_price"].sort_index()
week_1913 = calendar.loc[m5.TRAIN_END, "wm_yr_wk"]

rng = np.random.default_rng(0)
pick = rng.choice(len(X), 2000, replace=False)
mismatches = {}


def check(name, ok):
    mismatches[name] = mismatches.get(name, 0) + int(not ok)


for k in pick:
    r, H = rows[k], int(h[k])
    i, t0 = int(table.series[r]), int(table.t[r])
    first = int(np.flatnonzero(raw[i] > 0)[0]) + 1

    def seen(d):  # sales on day d as the model sees them
        return np.nan if d < first or d in christmas else raw[i, d - 1]

    x = X.iloc[k]
    check("h", x["h"] == H)
    lags = [seen(t0 - j) if t0 - j >= 1 else np.nan for j in range(F.N_LAGS)]
    check("lags", np.allclose(lags, x[F.LAG_COLS].to_numpy(float), equal_nan=True))
    check("target", raw[i, t0 + H - 1] == y[k])
    win = np.array([seen(d) for d in range(max(1, t0 - 27), t0 + 1)])
    check("mean_28", np.isclose(np.nanmean(win), x["mean_28"]))
    check("zero_share_28", np.isclose(np.mean(win[~np.isnan(win)] == 0), x["zero_share_28"]))
    check("days_since_sale", t0 - (np.flatnonzero(raw[i, :t0] > 0)[-1] + 1) == x["days_since_sale"])
    d = t0 + H
    event = calendar.loc[d, "event_name_1"]
    check("event", (event if pd.notna(event) else "none") == x["event"])
    check("snap", calendar.loc[d, f"snap_{sales.state_id.iloc[i]}"] == x["snap"])
    key = (sales.store_id.iloc[i], sales.item_id.iloc[i])
    p = prices.get((*key, calendar.loc[d, "wm_yr_wk"]), np.nan)
    history = prices.loc[key]
    median = history[history.index <= week_1913].median()
    check("rel_price", np.isclose(p / median, x["rel_price"]) if pd.notna(p) else np.isnan(x["rel_price"]))

assert sum(mismatches.values()) == 0, mismatches
pd.Series(mismatches, name="mismatches in 2,000 rows").to_frame()

,"mismatches in 2,000 rows"
h,0
lags,0
target,0
mean_28,0
zero_share_28,0
days_since_sale,0
event,0
snap,0
rel_price,0


**No look-ahead.** Groups L and S may use only days up to the origin. All sales after 27.03.2016. are replaced
by random numbers, and the inputs of every origin up to 27.03.2016. must stay exactly the same.

In [9]:
cut = m5.TRAIN_END - 28
scrambled = copy.copy(panel)
scrambled.y = panel.y.copy()
scrambled._agg = None
scrambled.y[:, cut + 1 :] = rng.poisson(5, size=scrambled.y[:, cut + 1 :].shape)
before = t <= cut
again = F.origin_table(scrambled, series[before], t[before])
same = np.array_equal(table.lags[before], again.lags, equal_nan=True) and table.s[before].reset_index(drop=True).equals(again.s)
assert same
print(f"L and S inputs of {before.sum():,} origins up to {m5.date_of(cut)} unchanged after replacing all later sales: {same}")

L and S inputs of 6,457,067 origins up to 27.03.2016. unchanged after replacing all later sales: True


**Group H.** The sales of the higher levels are computed from cumulative sums over the series of each level.
They are checked on 300 random rows against a direct sum over the series of the level, and with the same
look-ahead test as above.

In [10]:
pick = rng.choice(np.flatnonzero(before), 300, replace=False)
Xh, _, _ = F.design(panel, table, 1, ("H",), target=False, rows=pick)
worst = 0.0
for k, r in enumerate(pick):
    i, T = table.series[r], int(table.t[r])
    for lvl, cols in F.HIER_LEVELS.items():
        member = np.ones(len(panel.ids), dtype=bool)
        for c in cols:
            member &= (panel.hier[c] == panel.hier[c].iloc[i]).to_numpy()
        for w in F.HIER_WINDOWS:
            win = panel.y[member][:, max(T - w, 0) + 1 : T + 1]
            seen = ~np.isnan(win).all(axis=0)
            direct = np.nansum(win[:, seen]) / seen.sum()
            worst = max(worst, abs(direct - Xh[f"{lvl}_mean_{w}"].iloc[k]) / max(direct, 1))
scrambled._agg = None  # recompute the sales of the levels from the scrambled sales
Xs, _, _ = F.design(scrambled, table, 1, ("H",), target=False, rows=pick)
assert worst < 1e-4 and np.allclose(Xh.to_numpy(), Xs.to_numpy(), equal_nan=True)
print(f"group H, 300 rows: largest relative difference to the direct sums {worst:.1e}; unchanged after replacing all later sales: True")

group H, 300 rows: largest relative difference to the direct sums 5.1e-08; unchanged after replacing all later sales: True


## 6. Forecast rows

In the validation stage every series has one row for each of the 4 origins and 7 horizons. The rows of the
test stage need the test sales for the later origins and are built only in the final evaluation.

In [11]:
rows = []
for stage in ["valid"]:
    fs, ft = F.forecast_origins(panel, stage)
    ftable = F.origin_table(panel, fs, ft)
    frows, fh = F.horizon_rows(len(ftable))
    Xf, _, _ = F.design(panel, ftable, fh, target=False, rows=frows)
    rows.append({"stage": stage, "origins": ", ".join(m5.date_of(o) for o in np.unique(ft)), "rows": len(Xf),
                 "series with a missing lag": Xf[F.LAG_COLS].isna().any(axis=1).mean(),
                 "missing price on the target day": Xf["rel_price"].isna().mean()})
pd.DataFrame(rows).set_index("stage")

,origins,rows,series with a missing lag,missing price on the target day
stage,,,,
valid,"24.04.2016., 01.05.2016., 08.05.2016., 15.05.2...",853720,0.000,0.000


## Key findings

1. **Panel.** 30,490 series with sales 29.01.2011.–22.05.2016.; the test window stays sealed. 21.1% of the cells are
   missing on purpose: 20.9% before the first sale, 0.21% on Christmas Day.
2. **Inputs.** The horizon h and 88 inputs in five groups: 56 lags (L), 9 exogenous inputs known in advance (E), 9 sales-derived
   inputs (S), 5 hierarchy identifiers (ID) and 9 inputs from the sales of higher levels of the hierarchy (H). Availability is left out, because M5 never delists an item
   after its first price week.
3. **Size.** With weekly origins and all 7 horizons the matrix has about 46 million rows (about 15 GB), too
   many for the memory of a Kaggle GPU. With 2 random horizons per origin it has about 13 million rows
   (about 4 GB).
4. **Weekday.** With weekly origins every origin is a Sunday, so the target weekday `wday` and the horizon h
   carry the same information.
5. **Zeros.** 56–60% of the training targets are zero, which supports the Poisson and Tweedie losses.
6. **Checks.** 2,000 random rows recomputed from the raw CSV files match exactly, and the L, S and H inputs do not
   change when every later sale is replaced by random numbers.